# Lesson 33 — Piecewise-Linear Approximation and Grey-Box Models

## Learning objectives

1. Explain when a model should contain a **table** instead of a formula (pump curves, tariffs, property data), and why a piecewise-linear (PWL) table keeps the model mixed-integer *linear*.
2. Declare a tabulated function with `m.piecewise`, choose among its four exact MILP encodings (`incremental`, `log`, `disaggregated`, `sos2`), and predict their size from the number of breakpoints.
3. Use the domain rule (`PiecewiseDomainError`) to keep a table from silently clamping a model.
4. Distinguish a *declared* table (exact) from an *approximated* nonlinear term, and use `dm.nonlinear_to_pwl` in its two modes: a rigorous outer approximation that certifies, and a chord interpolant that never does.
5. Embed a **grey box**, a simulator that returns values and derivatives but no formula, with `dm.external`. Verify its derivatives, and state exactly what you can and cannot certify with it.

## 1. Tables, formulas, and boxes

Lessons 5–9 assumed that every function in a model is a formula `discopt` can read. Real engineering models often break that assumption in one of two ways.

- **The function is a table.** A pump datasheet gives power at a handful of flows, a utility tariff charges by consumption bands, and a property package returns tabulated enthalpies. Fitting a smooth formula through such data invents behaviour between the points. Interpolating linearly between them is the honest model, and a **piecewise-linear** function has an *exact* mixed-integer linear representation, so branch and bound certifies it like any other MILP {cite:p}`Vielma2010,Vielma2015`.
- **The function is a program.** A friction-factor solver, a flash calculation or a compiled reactor model returns a number, and often its derivatives, but no algebra. Such a *grey box* {cite:p}`Biegler2010` can sit inside a local NLP solve, but it has no convex relaxation, so it cannot be part of a global certificate {cite:p}`Eason2016`.

A third case connects the two: a nonlinear formula you would *like* to turn into a table, to get a MILP. That is an approximation, so it needs its own story about what is certified (Section 4).

| the function is… | tool | certificate |
|---|---|---|
| a table of breakpoints | `m.piecewise` (exact MILP encoding) | yes |
| a formula you choose to tabulate | `dm.nonlinear_to_pwl(mode="outer")` | yes, on the original model |
| ditto, chord interpolant | `dm.nonlinear_to_pwl(mode="approximate")` | never |
| a program with derivatives | `dm.external(fn, jac=, hess=)` | never (`feasible`, no bound) |

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm

## 2. Declaring a tabulated function

`y = m.piecewise(x, breakpoints, values, method=..., name=...)` returns a new variable $y$ tied to $x$ by mixed-integer linear rows, so that

$$y = f(x), \qquad f = \text{the linear interpolant of } (b_0, v_0), \dots, (b_{n-1}, v_{n-1}) \text{ on } [b_0, b_{n-1}].$$

The rows are an **exact** representation of the graph of $f$, at the breakpoints *and between them*. The model solved is the model declared, so `gap_certified=True` means what it always means. `values` may be an array or a callable sampled at the breakpoints, and an array `x` applies the same table elementwise.

**Example: pump scheduling.** Three identical pumps share a duty of 70 m³/h. Shaft power comes from a datasheet sampled at 17 flows. Efficiency peaks mid-range, so the power curve is *nonconvex*: two half-loaded pumps can beat one pump at full load, or lose to it. A running pump must deliver at least 8 m³/h and costs 1.5 kW in auxiliaries. Linking `q` to an on/off binary is exactly the Lesson 5 pattern.

In [ ]:
q_bp = np.linspace(0, 40, 17)                        # flow breakpoints, m3/h
eta = 0.78 * (1 - ((q_bp - 26) / 26) ** 2)           # efficiency, peaks at 26 m3/h
head = 55 - 0.012 * q_bp**2                          # pump head, m
P_tab = np.where(q_bp > 0,                           # shaft power, kW
                 9.81 * 1000 * (q_bp / 3600) * head / np.maximum(eta, 0.2) / 1000, 0.0)

def pump_model(method):
    m = do.Model("pumps_" + method)
    q = m.continuous("q", shape=(3,), lb=0, ub=40)
    on = m.binary("on", shape=(3,))
    P = m.piecewise(q, q_bp, P_tab, method=method, name="P")   # elementwise over 3 pumps
    m.subject_to(q <= 40 * on)
    m.subject_to(q >= 8 * on)
    m.subject_to(dm.sum(q) == 70)
    m.minimize(dm.sum(P) + 1.5 * dm.sum(on))
    return m, q

m, q = pump_model("incremental")
r = m.solve()
print(f"{r.status}, certified={r.gap_certified}, power = {r.objective:.4f} kW")
print("flows:", np.round(np.abs(r.value(q)), 3))

Two pumps at 35 m³/h draw less shaft power than three at 23.3 m³/h (about 11.2 kW against 12.2 kW, before the auxiliaries), and a single pump cannot carry 70 m³/h. Because the table is nonconvex, a local NLP on a smoothed fit could have stopped at any of these. The MILP proves which is best.

### 2.1 The four encodings

Every `method=` is exact. They differ in the number of binaries and auxiliary variables, and in how strong the LP relaxation is {cite:p}`Vielma2010`. For $n$ breakpoints ($n-1$ segments):

| `method` | binaries | continuous aux. | LP relaxation | reference |
|---|---|---|---|---|
| `"incremental"` (default) | $n-2$ | $n-1$ | locally ideal | {cite:p}`MarkowitzManne1957` |
| `"log"` | $\lceil \log_2(n-1)\rceil$ | $n$ | locally ideal | {cite:p}`Vielma2011` |
| `"disaggregated"` | $n-1$ | $2(n-1)$ | locally ideal | {cite:p}`Croxton2003` |
| `"sos2"` | $n$ (after lowering) | $n$ | weaker | {cite:p}`BealeTomlin1970,Dantzig1960` |

*Locally ideal* means the LP relaxation of the rows for one function is already the convex hull of its graph: every vertex is integral. `"log"` reaches this with only logarithmically many binaries by using a Gray code, in which adjacent segments differ in one bit. `"sos2"` declares a real SOS2 set (`m.sos2`, Lesson 5), which `discopt` lowers at solve time with one binary per breakpoint and $O(n^2)$ exclusion rows. The model writers (`.nl`, LP, MPS, GAMS) refuse SOS relations, so prefer one of the other three for a model you plan to export.

`m.num_integer` and `m.num_continuous` count the variables, so you can check the table directly. The input $x$ and the output $y$ account for 2 of the continuous variables.

In [ ]:
import time

print(f"{'method':>14} {'binaries':>9} {'cont. aux':>10} {'time (s)':>9}  objective")
for method in ["incremental", "log", "disaggregated", "sos2"]:
    m, q = pump_model(method)
    t0 = time.perf_counter()
    r = m.solve()
    dt = time.perf_counter() - t0
    # 3 functions; subtract the 3 on/off binaries and the 6 q, P columns
    print(f"{method:>14} {(m.num_integer - 3) // 3:9d} {(m.num_continuous - 6) // 3:10d} "
          f"{dt:9.2f}  {r.objective:.4f} ({r.status})")

All four agree on the certified optimum, as they must. The per-function counts match the table for $n = 17$. `"sos2"` reports 0 binaries because its SOS2 set is still a declaration at this point; it becomes binaries only when the model is lowered for the solve. On a problem this small the timings are all well under a second and mostly overhead. On longer tables and larger models, `"incremental"` and `"log"` are usually the fastest, and `"sos2"` falls far behind because of its quadratic number of exclusion rows. Exercise 1 asks you to predict the sizes before you count them.

### 2.2 The domain rule

Every encoding forces $b_0 \le x \le b_{n-1}$. If the declared domain of $x$ were wider than that, the table would **silently clamp** the model: the model solved would not be the one you wrote. `discopt` refuses with `PiecewiseDomainError`, both when you declare the table and, because bounds are mutable, again in `validate()` before every solve.

In [ ]:
m = do.Model("domain")
x = m.continuous("x", lb=0, ub=50)            # the table below stops at 40
try:
    m.piecewise(x, q_bp, P_tab)
except dm.PiecewiseDomainError as e:
    print("at declaration:", str(e)[:150], "...")

m = do.Model("domain2")
x = m.continuous("x", lb=0, ub=40)
y = m.piecewise(x, q_bp, P_tab)
m.minimize(y - 0.2 * x)
x.ub = np.float64(45.0)                        # widened afterwards
try:
    m.solve()
except dm.PiecewiseDomainError as e:
    print("at solve time  :", str(e)[:110], "...")

There are two honest fixes: give $x$ the bounds of the table, if flows above 40 are physically impossible, or extend the table, if they are possible. Repeated breakpoints (jumps) and multivariate tables are also refused. Model a jump with a disjunction (`m.either_or`, Lesson 23), and a two-input table with one piecewise call per input, or with a triangulation you write yourself {cite:p}`Vielma2010`.

## 3. A declared table is not an approximation

If the table was *sampled from* a nonlinear $g$, the model solved is the one with the interpolant of those samples, and the certificate is for that model, not for $g$. The chord of a concave $g$ lies below it, and the chord of a convex $g$ lies above it. So a PWL surrogate can cut off feasible points or admit infeasible ones, and its optimum is not a bound on the original problem in either direction.

In [ ]:
bp = np.linspace(0.0, 4.0, 5)
m = do.Model("sqrt_table")
x = m.continuous("x", lb=0.0, ub=4.0)
s = m.piecewise(x, bp, np.sqrt, name="sqrt_pwl")   # callable sampled at the breakpoints
m.maximize(s - 0.3 * x)
r = m.solve()
grid = np.linspace(0, 4, 400_001)
true_best = np.max(np.sqrt(grid) - 0.3 * grid)
print(f"PWL model : {r.objective:.4f} at x = {r.value(x):.3f}  (certified={r.gap_certified})")
print(f"true sqrt : {true_best:.4f}  -> the certified PWL optimum is not an upper bound")

## 4. Turning nonlinear terms into a MILP: `dm.nonlinear_to_pwl`

Sometimes you want a MILP on purpose: to hand the model to a MILP workflow, or to exploit separable structure. `T = dm.nonlinear_to_pwl(model, mode=..., segments=8)` finds every maximal **univariate** nonlinear subexpression of a bounded scalar input (an `exp(0.5*y)`, an `x**4 - 3*x**2 + x`) and replaces it on a **copy** of the model. The original is never modified. `T.terms` lists what was replaced. `T.skipped` lists univariate terms it could not tabulate, and why (an unbounded input, a `Parameter`, a range too large for meaningful coefficients). Multivariate terms such as `x*y` are left exactly as written, and `T.fully_linear` is then `False`.

**`mode="outer"` (default) is a rigorous relaxation.** On each segment $[a, b]$ with chord $s$, the new rows force $w - s(x)$ into a band $[e_\ell, e_u]$ that *provably* contains $g(x) - s(x)$. The band comes from interval evaluation of $g$ and mean-value bounds with $g'([a,b])$ enclosed by interval automatic differentiation {cite:p}`GriewankWalther2008`. Every point $(x, g(x))$ satisfies the rows, so the transformed MILP's bound is valid for the **original** model. `T.solve()` then verifies the MILP's point on the original model, polishes it with a local NLP, and refines the partition where the point strayed. This is the adaptive-partitioning idea of {cite:p}`Misener2011,Nagarajan2019`. The loop stops when the verified incumbent meets the bound (`optimal`, `gap_certified=True`), or when `max_rounds` or `max_breakpoints` runs out (`feasible`, still with a valid bound).

**`mode="approximate"` is the chord interpolant.** It is cheap, and **never** certified: `bound=None`, `gap_certified=False`, and `feasible` only if the point checks out on the original model.

The cell below solves the same nonconvex model in three ways. (The outer bands use coefficients that span many orders of magnitude near the breakpoints, which trips `discopt`'s scaling diagnostic. That warning concerns rows the transformation built itself, so we silence it here.)

In [ ]:
import warnings

def smooth_model():
    m = do.Model("smooth")
    x = m.continuous("x", lb=-2, ub=2)
    y = m.continuous("y", lb=-1, ub=3)
    m.minimize(x**4 - 3 * x**2 + x + dm.exp(0.5 * y) - y)
    m.subject_to(x + y >= 0.5)
    return m, x, y

m, x, y = smooth_model()
T = dm.nonlinear_to_pwl(m)                      # mode="outer", segments=8
print(T)
for term in T.terms:
    print(f"  {term.expression:28s} input={term.input}  breakpoints={len(term.breakpoints)}")

t0 = time.perf_counter()
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Badly scaled model")
    r_out = T.solve(time_limit=60)
t_out = time.perf_counter() - t0
st = r_out.solver_stats["nonlinear_to_pwl"]
print(f"outer      : {r_out.status:8s} obj={r_out.objective:.6f} bound={r_out.bound:.6f} "
      f"certified={r_out.gap_certified} rounds={st['rounds']} ({t_out:.1f} s)")

r_app = dm.nonlinear_to_pwl(m, mode="approximate").solve()
print(f"approximate: {r_app.status:8s} obj={r_app.objective:.6f} bound={r_app.bound} "
      f"certified={r_app.gap_certified}")

t0 = time.perf_counter()
r_glob = m.solve()
print(f"discopt    : {r_glob.status:8s} obj={r_glob.objective:.6f} bound={r_glob.bound:.6f} "
      f"({time.perf_counter() - t0:.1f} s)")
print(f"breakpoints after refinement: {[len(tt.breakpoints) for tt in T.terms]}")

All three find the same point. Only the outer mode and `discopt`'s own spatial branch and bound *prove* it optimal, and the spatial B&B is much faster. It relaxes the same terms directly (Lesson 21), without partition binaries. `st["history"]` shows the outer mode's bound rising round by round as the refinement adds breakpoints. This is the measured pattern on `discopt`'s test corpus too: the outer mode never produced an invalid bound, and it was slower than the default solver on nearly every instance. Use it when you *want* a MILP, not to go faster.

**A harder case.** Economic dispatch with valve-point loading {cite:p}`Walters1993` adds a rectified sine ripple $|e_i \sin(f_i (P_i^{\min} - P_i))|$ to each generator's quadratic cost. The kinks make the problem nonsmooth and highly multimodal. Within a short time budget both routes return a verified dispatch and a **valid** bound that brackets the literature optimum (8234.07). Neither closes the gap, and `discopt`'s own bound is the much tighter of the two:

In [ ]:
import contextlib, io
from discopt.modeling import examples

with contextlib.redirect_stdout(io.StringIO()):
    md_ = examples.example_valve_point_dispatch()
P = next(v for v in md_._variables if v.name == "P")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Badly scaled model")
    r_vp = dm.nonlinear_to_pwl(md_, segments=8).solve(time_limit=10)
print(f"outer PWL : {r_vp.status:8s} cost={r_vp.objective:.2f} bound={r_vp.bound:.2f} "
      f"gap={100 * r_vp.gap:.2f}%  dispatch={np.round(r_vp.value(P), 1)}")
r_sb = md_.solve(time_limit=10)
print(f"discopt   : {r_sb.status:8s} cost={r_sb.objective:.2f} bound={r_sb.bound:.2f} "
      f"gap={100 * r_sb.gap:.2f}%")
print("literature optimum 8234.07 inside both [bound, cost]:",
      r_vp.bound <= 8234.07 <= r_vp.objective and r_sb.bound <= 8234.07 <= r_sb.objective)

A `feasible` result with a bound is still useful information: it tells you how far from optimal the verified point can be. An `approximate`-mode result on the same model gives you no such number.

## 5. Grey boxes: `dm.external`

A grey box is a block whose value comes from code. Our example is a continuous stirred-tank reactor with second-order kinetics $A \to B$. The conversion $X$ at residence time $\tau$ (h) and temperature $T$ (K) solves the steady-state balance

$$G(X; \tau, T) = k(T)\, \tau\, C_{A0}\, (1 - X)^2 - X = 0, \qquad k(T) = k_0 e^{-E/RT}.$$

A real flowsheet would call a simulator here. We solve $G = 0$ by Newton's method, which is a loop with a data-dependent exit and cannot be traced, and we get derivatives from the **implicit function theorem** applied to the converged residual {cite:p}`GriewankWalther2008`. Writing $a = k(T)\,\tau\, C_{A0}$:

$$\frac{\partial X}{\partial p} = -\frac{\partial G/\partial p}{\partial G/\partial X}, \qquad \frac{\partial G}{\partial X} = -2a(1-X) - 1, \quad \frac{\partial G}{\partial \tau} = k C_{A0}(1-X)^2, \quad \frac{\partial G}{\partial T} = a\,\frac{E/R}{T^2}(1-X)^2.$$

`dm.external(fn, jac=, hess=, shape=)` wraps it. Each callable receives a plain `numpy` array: `fn` returns shape `shape`, `jac` returns `shape + x.shape`, and `hess` returns `shape + x.shape + x.shape`. Here we form the Hessian by central differences of our exact Jacobian. A Newton-type NLP solver tolerates a slightly inexact Hessian, at some cost in convergence rate, but it does not tolerate a wrong Jacobian.

In [ ]:
import logging
# Every grey-box solve logs that it found no dual bound (and that FBBT cannot read the
# opaque node). We print r.bound instead, so keep discopt's log to errors.
logging.getLogger("discopt").setLevel(logging.ERROR)

k0, EoR, CA0 = 4.0e6, 6000.0, 2.0      # L/(mol h), K, mol/L
CALLS = {"fn": 0, "jac": 0, "hess": 0}

def cstr_solve(tau, T):
    k = k0 * np.exp(-EoR / T)
    a = k * tau * CA0
    X = 0.5
    for _ in range(100):                                # Newton on G(X) = 0
        step = (a * (1 - X) ** 2 - X) / (-2 * a * (1 - X) - 1)
        X -= step
        if abs(step) < 1e-15:
            break
    GX = -2 * a * (1 - X) - 1
    dX = -np.array([k * CA0 * (1 - X) ** 2, a * EoR / T**2 * (1 - X) ** 2]) / GX
    return X, dX

def fn(z):
    CALLS["fn"] += 1
    return np.array([cstr_solve(z[0], z[1])[0]])

def jac(z):
    CALLS["jac"] += 1
    return cstr_solve(z[0], z[1])[1][None, :]                # shape (1,) + (2,)

def hess(z):
    CALLS["hess"] += 1
    z = np.asarray(z, dtype=float)
    H = np.zeros((1, 2, 2))                                  # shape (1,) + (2,) + (2,)
    for j in range(2):
        e = np.zeros(2)
        e[j] = 1e-6 * max(1.0, abs(z[j]))
        H[0, :, j] = (cstr_solve(*(z + e))[1] - cstr_solve(*(z - e))[1]) / (2 * e[j])
    return 0.5 * (H + H.transpose(0, 2, 1))

cstr = dm.external(fn, jac=jac, hess=hess, shape=(1,), name="cstr")

# Check the derivatives against finite differences BEFORE building a model.
z0, h = np.array([2.0, 350.0]), 1e-6
J_fd = np.array([(fn(z0 + h * e)[0] - fn(z0 - h * e)[0]) / (2 * h) for e in np.eye(2)])
print("X(z0) =", fn(z0)[0], "  jac =", jac(z0)[0], "  max rel. err vs FD:",
      np.max(np.abs(J_fd - jac(z0)[0]) / np.abs(jac(z0)[0])))

`discopt` checks the *shapes* your callables return, but it cannot check their *values*. A Jacobian with the right shape and the wrong numbers silently sends the solver the wrong way. Always compare against finite differences.

Now design the reactor. Product sells at 100 per unit conversion, residence time costs 4 per hour (reactor size), and heating costs 0.15 per kelvin above 300 K:

$$\max_{\tau \in [0.1, 10],\; T \in [300, 400]} \; 100\,X(\tau, T) - 4\tau - 0.15\,(T - 300).$$

In [ ]:
def build_cstr(grey_box=True):
    m = do.Model("cstr_" + ("ext" if grey_box else "sym"))
    z = m.continuous("z", shape=(2,), lb=[0.1, 300.0], ub=[10.0, 400.0])   # (tau, T)
    if grey_box:
        X = cstr(z)[0]
    else:   # the symbolic twin: X is a variable and G = 0 a constraint
        X = m.continuous("X", lb=0, ub=1)
        m.subject_to(k0 * dm.exp(-EoR / z[1]) * z[0] * CA0 * (1 - X) ** 2 == X)
    m.maximize(100 * X - 4 * z[0] - 0.15 * (z[1] - 300))
    return m, z

for k in CALLS:
    CALLS[k] = 0
m, z = build_cstr()
r = m.solve()
print(f"grey box, default start : {r.status:8s} profit={r.objective:9.4f}  (tau, T)={np.round(r.value(z), 4)}")
print(f"  bound={r.bound}  gap_certified={r.gap_certified}  calls={CALLS}")

The solve stopped at the corner $(\tau, T) = (0.1, 300)$ with a *negative* profit. That corner is a genuine local maximum: the reaction is too slow there for a small step in $\tau$ or $T$ to pay for itself. A local NLP started at the lower bounds, which is the default start here, has no way to know that a much better plant exists. Give it other starting points:

In [ ]:
for start in ([1.0, 350.0], [5.0, 330.0], [8.0, 380.0]):
    m, z = build_cstr()
    r = m.solve(initial_solution={z: np.array(start)})
    print(f"start {start}: {r.status:8s} profit={r.objective:.6f}  (tau, T)={np.round(r.value(z), 4)}")

m_sym, z_sym = build_cstr(grey_box=False)
r_sym = m_sym.solve()
print(f"symbolic twin          : {r_sym.status:8s} profit={r_sym.objective:.6f}  bound={r_sym.bound:.6f}  "
      f"certified={r_sym.gap_certified}")

Multistart finds 42.291, and here we can *prove* it is the global optimum, because this simulator happens to have a symbolic twin: make $X$ a variable and $G = 0$ a constraint, and `discopt` certifies the whole model. A real simulator gives you no such twin. That is what you give up with a grey box.

### 5.1 The certificate contract

- **No bound, ever.** A global bound needs a convex relaxation of every function in the model {cite:p}`Tawarmalani2005`, and a function you can only *evaluate* admits none: between two samples it could do anything. So a model with `dm.external` reports `status="feasible"`, `bound=None` and `gap_certified=False`, never `optimal`. That is an honest description of a local solve.
- **No integer variables.** Branch and bound would need a valid relaxation at every node to prune soundly. `discopt` refuses the combination with a `ValueError` instead of pruning on nothing. The message calls the block a `dm.custom` function, because `dm.external` reuses that node type. The refusal includes the binaries a `m.piecewise` table adds (Exercise 5).
- **`jac` is required, `hess` is needed for the NLP path.** Without `hess`, `solve()` refuses and names the two ways out: supply it, or use `solver="direct"`, a derivative-free sampling search {cite:p}`Conn2009,Larson2019`.
- **Calls are memoized.** Each callable is cached on the exact input bytes, so a point the solver revisits is not recomputed. The cache is cleared at the start of every solve. The counters above show how many distinct evaluations a local solve needed.
- **A pure black box** (values only, no derivatives) is `dm.custom(f)` with `solve(solver="direct")`. Its body must be JAX-traceable, which needs the `jax` extra.

In [ ]:
for k in CALLS:
    CALLS[k] = 0
m, z = build_cstr()
r_d = m.solve(solver="direct", max_evals=300)
print(f"solver='direct': {r_d.status} profit={r_d.objective:.6f} (tau, T)={np.round(r_d.value(z), 4)} "
      f"bound={r_d.bound}  calls={CALLS}")

m_int = do.Model("with_integer")
zz = m_int.continuous("z", shape=(2,), lb=[0.1, 300.0], ub=[10.0, 400.0])
n_trains = m_int.integer("n", lb=1, ub=3)
m_int.maximize(100 * cstr(zz)[0] - 4 * zz[0] * n_trains)
try:
    m_int.solve()
except ValueError as e:
    print("with an integer variable:", str(e)[:110], "...")

The derivative-free search found the same plant, at the price of many more evaluations, and it still has no bound. On an expensive simulator that difference is decisive. Trust-region filter methods {cite:p}`Eason2016` sit between the two: they build local surrogates of the grey box and still guarantee convergence to a local optimum of the true model.

## 6. Practical checklist

1. **Table or formula?** If the data *are* the model, use `m.piecewise`. If a formula exists, keep it and let `discopt` relax it. Use `nonlinear_to_pwl` only when you need a MILP.
2. **Bounds first.** Give every piecewise input exactly the table's span, or extend the table. Never widen a bound after declaring a table.
3. **Encoding.** Start with `"incremental"`, try `"log"` on long tables, and avoid `"sos2"` for long tables and for models you will export.
4. **Read the mode.** Only `mode="outer"` gives a bound. Report `r.status` and `r.bound`, not just the objective.
5. **Grey boxes: verify derivatives** against finite differences, at several points.
6. **Grey boxes: multistart**, and report the result as `feasible`. If any part of the block can be written symbolically, a symbolic twin is the only route to a certificate.
7. **Keep integers out of grey-box models.** Fix the discrete decisions and enumerate them, use a symbolic twin, or move the logic outside the solve.

## References
{cite:p}`Vielma2010,Vielma2011,Vielma2015,Misener2011,Walters1993,Biegler2010,Eason2016,Conn2009`